In [1]:
import pandas as pd
import numpy as np
import re
import string
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from scipy.sparse import hstack, csr_matrix

import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully!")

All libraries imported successfully!


In [13]:
import re
import numpy as np
import pandas as pd

def extract_features(message):

    message = str(message)
    words = message.split()
    lower_message = message.lower()

    features = {
        "message_length": len(message),
        "word_count": len(words),

        "avg_word_length": (
            np.mean([len(w) for w in words]) if words else 0
        ),

        "unique_word_ratio": (
            len(set(words)) / len(words) if words else 0
        ),

        "digit_count": sum(c.isdigit() for c in message),

        "digit_ratio": (
            sum(c.isdigit() for c in message) / len(message)
            if message else 0
        ),

        "uppercase_count": sum(c.isupper() for c in message),

        "uppercase_ratio": (
            sum(c.isupper() for c in message) / len(message)
            if message else 0
        ),

        "special_char_count": sum(
            not c.isalnum() and not c.isspace()
            for c in message
        ),

        "special_char_ratio": (
            sum(not c.isalnum() and not c.isspace()
                for c in message) / len(message)
            if message else 0
        ),

        "space_count": message.count(" "),
        "exclamation_count": message.count("!"),
        "question_count": message.count("?"),

        "currency_symbol_count": sum(
            c in "$€£₹" for c in message
        ),

        "percent_count": message.count("%"),

        "has_all_caps_word": int(
            any(w.isupper() and len(w) > 1 for w in words)
        ),

        "repeated_punctuation_count": len(
            re.findall(r"[!?.,]{2,}", message)
        ),

        "repeated_character_count": len(
            re.findall(r"(.)\1{2,}", lower_message)
        ),

        "leet_obfuscation_count": len(
            re.findall(r"[a-zA-Z][013457@$][a-zA-Z]", message)
        ),

        "url_count": len(
            re.findall(r"https?://|www\.", lower_message)
        ),

        "has_url": int(
            bool(re.search(r"https?://|www\.", lower_message))
        ),

        "url_char_count": sum(
            len(url) for url in re.findall(
                r"(?:https?://|www\.)\S+", lower_message
            )
        ),

        "avg_url_length": np.mean([
            len(url) for url in re.findall(
                r"(?:https?://|www\.)\S+", lower_message
            )
        ]) if re.findall(r"(?:https?://|www\.)\S+", lower_message) else 0,

        "shortened_url_count": len(
            re.findall(
                r"bit\.ly|tinyurl|t\.co|goo\.gl|is\.gd|ow\.ly",
                lower_message
            )
        ),

        "suspicious_tld_count": len(
            re.findall(
                r"\.(xyz|top|click|link|work|info|online)\b",
                lower_message
            )
        ),

        "ip_address_count": len(
            re.findall(
                r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
                lower_message
            )
        ),

        "has_ip_address": int(
            bool(re.search(
                r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
                lower_message
            ))
        ),

        "email_count": len(
            re.findall(r"\b[\w.+-]+@[\w.-]+\.\w+\b", message)
        ),

        "has_email": int("@" in message),

        "phone_count": len(
            re.findall(r"\+?\d[\d\s-]{8,}\d", message)
        ),

        "has_phone": int(
            bool(re.search(r"\+?\d[\d\s-]{8,}\d", message))
        )
    }

    keyword_groups = {
        "urgent": [
            "urgent", "immediately", "hurry", "limited time",
            "expire", "suspended", "blocked", "action required"
        ],

        "financial": [
            "bank", "account", "payment", "refund", "money",
            "cash", "debit", "credit", "transaction"
        ],

        "credential": [
            "password", "otp", "pin", "login", "verify",
            "username", "credentials", "security code"
        ],

        "action": [
            "click", "update", "confirm", "verify",
            "register", "submit", "visit", "reply"
        ],

        "reward": [
            "won", "winner", "prize", "reward",
            "lottery", "free", "claim", "gift"
        ],

        "delivery": [
            "delivery", "parcel", "package", "shipment",
            "courier", "tracking", "dispatch"
        ]
    }

    for group, keywords in keyword_groups.items():

        count = sum(
            lower_message.count(keyword)
            for keyword in keywords
        )

        features[group + "_keyword_count"] = count

        features["has_" + group + "_language"] = int(count > 0)

    suspicious_keywords = (
        keyword_groups["urgent"]
        + keyword_groups["credential"]
        + keyword_groups["reward"]
    )

    suspicious_count = sum(
        lower_message.count(keyword)
        for keyword in suspicious_keywords
    )

    features["suspicious_keyword_count"] = suspicious_count

    return features


print("Feature extraction function created successfully!")

Feature extraction function created successfully!


In [164]:
import pandas as pd

# Training dataset
df_train_new = pd.read_csv(
    "ExAIS_SMS_feature_extracted_balanced_48cols.csv"
)

# External testing dataset
df_external_new = pd.read_csv(
    "sms_feature_extracted.csv"
)

print("TRAINING DATASET SHAPE:", df_train_new.shape)
print("EXTERNAL DATASET SHAPE:", df_external_new.shape)

print("\nTRAINING DATASET COLUMNS:")
print(df_train_new.columns.tolist())

print("\nEXTERNAL DATASET COLUMNS:")
print(df_external_new.columns.tolist())

TRAINING DATASET SHAPE: (2780, 48)
EXTERNAL DATASET SHAPE: (10191, 48)

TRAINING DATASET COLUMNS:
['LABEL', 'TEXT', 'message_length', 'word_count', 'avg_word_length', 'unique_word_ratio', 'digit_count', 'digit_ratio', 'uppercase_count', 'uppercase_ratio', 'special_char_count', 'special_char_ratio', 'space_count', 'exclamation_count', 'question_count', 'currency_symbol_count', 'percent_count', 'has_all_caps_word', 'repeated_punctuation_count', 'repeated_character_count', 'leet_obfuscation_count', 'url_count', 'has_url', 'url_char_count', 'avg_url_length', 'shortened_url_count', 'suspicious_tld_count', 'ip_address_count', 'has_ip_address', 'email_count', 'has_email', 'phone_count', 'has_phone', 'urgent_keyword_count', 'financial_keyword_count', 'credential_keyword_count', 'action_keyword_count', 'reward_keyword_count', 'delivery_keyword_count', 'suspicious_keyword_count', 'has_urgent_language', 'has_financial_language', 'has_credential_language', 'has_action_language', 'has_reward_langua

In [165]:
print("TRAINING DATASET")
print("----------------")
print("\nLabel distribution:")
print(df_train_new["LABEL"].value_counts())

print("\nMissing values:", df_train_new.isnull().sum().sum())
print("Duplicate rows:", df_train_new.duplicated().sum())
print("Unique SMS messages:", df_train_new["TEXT"].nunique())


print("\n\nEXTERNAL DATASET")
print("----------------")
print("\nLabel distribution:")
print(df_external_new["LABEL"].value_counts())

print("\nMissing values:", df_external_new.isnull().sum().sum())
print("Duplicate rows:", df_external_new.duplicated().sum())
print("Unique SMS messages:", df_external_new["TEXT"].nunique())

TRAINING DATASET
----------------

Label distribution:
LABEL
spam    1390
ham     1390
Name: count, dtype: int64

Missing values: 0
Duplicate rows: 0
Unique SMS messages: 2780


EXTERNAL DATASET
----------------

Label distribution:
LABEL
ham         3397
smishing    3397
spam        3397
Name: count, dtype: int64

Missing values: 0
Duplicate rows: 2212
Unique SMS messages: 7931


In [124]:
# Normalize SMS text for comparison
df_train_new["TEXT_CLEAN"] = (
    df_train_new["TEXT"].astype(str).str.lower().str.strip()
)

df_external_new["TEXT_CLEAN"] = (
    df_external_new["TEXT"].astype(str).str.lower().str.strip()
)

# Map training labels
df_train_new["target"] = df_train_new["LABEL"].map({
    "ham": 0,
    "spam": 1
})

# Keep only ham and smishing in external dataset
df_external_binary = df_external_new[
    df_external_new["LABEL"].isin(["ham", "smishing"])
].copy()

df_external_binary["target"] = df_external_binary["LABEL"].map({
    "ham": 0,
    "smishing": 1
})

# Check overlap
train_messages = set(df_train_new["TEXT_CLEAN"])
external_messages = set(df_external_binary["TEXT_CLEAN"])

overlap = train_messages.intersection(external_messages)

print("TRAINING DATA")
print("Shape:", df_train_new.shape)
print(df_train_new["target"].value_counts())

print("\nEXTERNAL BINARY DATA")
print("Shape:", df_external_binary.shape)
print(df_external_binary["target"].value_counts())

print("\nOVERLAP CHECK")
print("Matching messages:", len(overlap))
print("External messages:", len(external_messages))
print("Unique external messages:", len(external_messages - train_messages))

TRAINING DATA
Shape: (2780, 50)
target
1    1390
0    1390
Name: count, dtype: int64

EXTERNAL BINARY DATA
Shape: (6794, 50)
target
0    3397
1    3397
Name: count, dtype: int64

OVERLAP CHECK
Matching messages: 0
External messages: 5768
Unique external messages: 5768


In [125]:
print("TRAINING - HAM EXAMPLES")
print("-----------------------")
print(df_train_new[df_train_new["LABEL"] == "ham"]["TEXT"].head(10).to_string(index=False))

print("\nTRAINING - SPAM EXAMPLES")
print("------------------------")
print(df_train_new[df_train_new["LABEL"] == "spam"]["TEXT"].head(10).to_string(index=False))

print("\nEXTERNAL - SMISHING EXAMPLES")
print("----------------------------")
print(df_external_new[df_external_new["LABEL"] == "smishing"]["TEXT"].head(10).to_string(index=False))

TRAINING - HAM EXAMPLES
-----------------------
Hmmm.keep posting me o. I'll also have money o....
             Daniel mercy oluwaseun 0050967585 gtb
Dear customer, you have used 25% out of your ti...
Your Acct 304XXXX489 Has Been Debited with NGN2...
Acct: ******1736Amt: NGN5,000.00 DrDesc: CASH W...
I'll collect it later jare... don't disturb urs...
Enjoy more Airtime on your Mobile phone today. ...
13/02/2014 21:01,153,We're sorry.D church we wa...
Stay blessed today. For Biblical Quotes text 1 ...
SmartStart Certificatn Session begins @ our nxt...

TRAINING - SPAM EXAMPLES
------------------------
Medical: Gall stones can cause a right-sided tu...
How Nigerian are you? Prove it! Answer 3 Questi...
Your MTN Smart Biz will expire on 05/Sep/2014 1...
200.00NGN has been credited. Main Bal:N 200.88....
Your account was credited with N100 and service...
Within you lies the power to do things you neve...
One who creates the opportunity is Brilliant. O...
FREE FOR 5 DAYS! Get all football 

In [126]:
print("TRAINING FEATURE DATA TYPES")
print(df_train_new.drop(
    columns=["LABEL", "TEXT", "TEXT_CLEAN", "target"]
).dtypes.value_counts())

print("\nNON-NUMERIC TRAINING COLUMNS")
print(
    df_train_new.drop(
        columns=["LABEL", "TEXT", "TEXT_CLEAN", "target"]
    ).select_dtypes(exclude="number").columns.tolist()
)

print("\nEXTERNAL FEATURE DATA TYPES")
print(df_external_binary.drop(
    columns=["LABEL", "TEXT", "TEXT_CLEAN", "target"]
).dtypes.value_counts())

TRAINING FEATURE DATA TYPES
int64      40
float64     6
Name: count, dtype: int64

NON-NUMERIC TRAINING COLUMNS
[]

EXTERNAL FEATURE DATA TYPES
int64      40
float64     6
Name: count, dtype: int64


In [127]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate text and labels
X_text = df_train_new["TEXT_CLEAN"]
y = df_train_new["target"]

# Numerical features
feature_columns = [
    col for col in df_train_new.columns
    if col not in ["LABEL", "TEXT", "TEXT_CLEAN", "target"]
]

X_numeric = df_train_new[feature_columns].astype(float)

# Train-test split
X_train_text, X_test_text, y_train, y_test, X_train_num, X_test_num = train_test_split(
    X_text,
    y,
    X_numeric,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# TF-IDF
tfidf_new = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_text_vec = tfidf_new.fit_transform(X_train_text)
X_test_text_vec = tfidf_new.transform(X_test_text)

# Scale numerical features
scaler_new = StandardScaler()

X_train_num_scaled = scaler_new.fit_transform(X_train_num)
X_test_num_scaled = scaler_new.transform(X_test_num)

# Combine features
X_train_final = hstack([
    X_train_text_vec,
    X_train_num_scaled
]).tocsr()

X_test_final = hstack([
    X_test_text_vec,
    X_test_num_scaled
]).tocsr()

print("FEATURE PREPARATION COMPLETED")
print("Training shape:", X_train_final.shape)
print("Testing shape:", X_test_final.shape)

FEATURE PREPARATION COMPLETED
Training shape: (2224, 5046)
Testing shape: (556, 5046)


In [128]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

models_new = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=1
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "SVM": LinearSVC(
        C=1,
        class_weight="balanced",
        max_iter=10000,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

for name, model in models_new.items():
    print("Training:", name)
    model.fit(X_train_final, y_train)

print("\nALL FOUR MODELS TRAINED SUCCESSFULLY!")

Training: Logistic Regression
Training: Random Forest
Training: SVM
Training: XGBoost

ALL FOUR MODELS TRAINED SUCCESSFULLY!


In [129]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

results_new = []

for name, model in models_new.items():

    y_pred = model.predict(X_test_final)

    results_new.append({
        "Model": name,
        "Accuracy (%)": round(accuracy_score(y_test, y_pred) * 100, 2),
        "Precision (%)": round(precision_score(y_test, y_pred) * 100, 2),
        "Recall (%)": round(recall_score(y_test, y_pred) * 100, 2),
        "F1 Score (%)": round(f1_score(y_test, y_pred) * 100, 2)
    })

results_new_df = pd.DataFrame(results_new)

print("INTERNAL TEST RESULTS")
display(results_new_df)

INTERNAL TEST RESULTS


,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%)
0,Logistic Regression,85.07,83.97,86.69,85.31
1,Random Forest,88.31,85.86,91.73,88.70
2,SVM,89.39,88.69,90.29,89.48
3,XGBoost,86.51,84.64,89.21,86.87


In [130]:
# External text features
X_external_text_new = df_external_binary["TEXT_CLEAN"]

X_external_text_vec_new = tfidf_new.transform(
    X_external_text_new
)

# External numerical features
X_external_num_new = df_external_binary[feature_columns].astype(float)

X_external_num_scaled_new = scaler_new.transform(
    X_external_num_new
)

# Combine text and numerical features
X_external_final_new = hstack([
    X_external_text_vec_new,
    X_external_num_scaled_new
]).tocsr()

# External labels
y_external_new = df_external_binary["target"]

print("EXTERNAL FEATURE PREPARATION COMPLETED")
print("External feature shape:", X_external_final_new.shape)
print("External labels shape:", y_external_new.shape)
print("\nExternal label distribution:")
print(y_external_new.value_counts())

EXTERNAL FEATURE PREPARATION COMPLETED
External feature shape: (6794, 5046)
External labels shape: (6794,)

External label distribution:
target
0    3397
1    3397
Name: count, dtype: int64


In [131]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

external_results_new = []

for name, model in models_new.items():

    y_pred = model.predict(X_external_final_new)

    external_results_new.append({
        "Model": name,
        "Accuracy (%)": round(
            accuracy_score(y_external_new, y_pred) * 100, 2
        ),
        "Precision (%)": round(
            precision_score(y_external_new, y_pred) * 100, 2
        ),
        "Recall (%)": round(
            recall_score(y_external_new, y_pred) * 100, 2
        ),
        "F1 Score (%)": round(
            f1_score(y_external_new, y_pred) * 100, 2
        )
    })

external_results_new_df = pd.DataFrame(external_results_new)

print("FINAL EXTERNAL TEST RESULTS")
display(external_results_new_df)

FINAL EXTERNAL TEST RESULTS


,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%)
0,Logistic Regression,68.03,77.45,50.87,61.41
1,Random Forest,78.94,85.95,69.18,76.66
2,SVM,61.05,66.33,44.89,53.55
3,XGBoost,80.62,84.71,74.71,79.40


In [132]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_xgb = models_new["XGBoost"].predict(
    X_external_final_new
)

print("XGBOOST EXTERNAL CONFUSION MATRIX")
print(confusion_matrix(y_external_new, y_pred_xgb))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_external_new,
    y_pred_xgb,
    target_names=["Legitimate", "Phishing"]
))

XGBOOST EXTERNAL CONFUSION MATRIX
[[2939  458]
 [ 859 2538]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Legitimate       0.77      0.87      0.82      3397
    Phishing       0.85      0.75      0.79      3397

    accuracy                           0.81      6794
   macro avg       0.81      0.81      0.81      6794
weighted avg       0.81      0.81      0.81      6794



In [140]:
variables_to_check = [
    "word_vectorizer",
    "char_vectorizer",
    "scaler_new",
    "feature_columns",
    "models_improved",
    "X_train_improved"
]

for var in variables_to_check:
    print(var, ":", var in globals())

word_vectorizer : False
char_vectorizer : False
scaler_new : True
feature_columns : True
models_improved : False
X_train_improved : False


In [141]:
import pandas as pd
import numpy as np
import re
import string
from scipy.sparse import hstack

def extract_sms_features(message):

    words = message.split()
    lower = message.lower()

    urls = re.findall(r'https?://\S+|www\.\S+', message)
    digits = sum(c.isdigit() for c in message)
    uppercase = sum(c.isupper() for c in message)
    special = sum(c in string.punctuation for c in message)

    urgent = ["urgent", "immediately", "now", "alert", "limited"]
    financial = ["bank", "account", "payment", "money", "refund"]
    credential = ["password", "otp", "pin", "verify", "login"]
    action = ["click", "call", "claim", "update", "confirm"]
    reward = ["win", "winner", "prize", "free", "reward"]
    delivery = ["delivery", "parcel", "package", "shipment"]
    suspicious = ["suspended", "blocked", "expire", "congratulations"]

    def count(items):
        return sum(lower.count(w) for w in items)

    def has(items):
        return int(any(w in lower for w in items))

    url_lengths = [len(u) for u in urls]

    email_count = len(re.findall(r'\b[\w.-]+@[\w.-]+\.\w+\b', message))
    phone_count = len(re.findall(r'\b\d{7,15}\b', message))
    ip_count = len(re.findall(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', message))

    features = {
        "message_length": len(message),
        "word_count": len(words),
        "avg_word_length": np.mean([len(w) for w in words]) if words else 0,
        "unique_word_ratio": len(set(words)) / len(words) if words else 0,
        "digit_count": digits,
        "digit_ratio": digits / len(message) if message else 0,
        "uppercase_count": uppercase,
        "uppercase_ratio": uppercase / len(message) if message else 0,
        "special_char_count": special,
        "special_char_ratio": special / len(message) if message else 0,
        "space_count": message.count(" "),
        "exclamation_count": message.count("!"),
        "question_count": message.count("?"),
        "currency_symbol_count": sum(c in "$£€₹" for c in message),
        "percent_count": message.count("%"),
        "has_all_caps_word": int(any(w.isupper() and len(w) > 1 for w in words)),
        "repeated_punctuation_count": len(re.findall(r'([!?.,])\1+', message)),
        "repeated_character_count": len(re.findall(r'(.)\1{2,}', message)),
        "leet_obfuscation_count": sum(lower.count(x) for x in ["0", "1", "3", "4", "5", "@"]),
        "url_count": len(urls),
        "has_url": int(bool(urls)),
        "url_char_count": sum(url_lengths),
        "avg_url_length": np.mean(url_lengths) if urls else 0,
        "shortened_url_count": sum(any(d in u.lower() for d in ["bit.ly", "tinyurl", "t.co", "goo.gl"]) for u in urls),
        "suspicious_tld_count": sum(any(t in u.lower() for t in [".xyz", ".top", ".click", ".loan"]) for u in urls),
        "ip_address_count": ip_count,
        "has_ip_address": int(ip_count > 0),
        "email_count": email_count,
        "has_email": int(email_count > 0),
        "phone_count": phone_count,
        "has_phone": int(phone_count > 0),
        "urgent_keyword_count": count(urgent),
        "financial_keyword_count": count(financial),
        "credential_keyword_count": count(credential),
        "action_keyword_count": count(action),
        "reward_keyword_count": count(reward),
        "delivery_keyword_count": count(delivery),
        "suspicious_keyword_count": count(suspicious),
        "has_urgent_language": has(urgent),
        "has_financial_language": has(financial),
        "has_credential_language": has(credential),
        "has_action_language": has(action),
        "has_reward_language": has(reward),
        "url_field_yes": int(bool(urls)),
        "email_field_yes": int(email_count > 0),
        "phone_field_yes": int(phone_count > 0)
    }

    return pd.DataFrame([features])


def test_sms_message():

    print("=" * 55)
    print("MULTI-MODEL SMS PHISHING DETECTION")
    print("=" * 55)

    sms = input("\nEnter SMS message: ").strip()

    if not sms:
        print("Please enter a message.")
        return

    sms_clean = sms.lower().strip()

    # Extract numerical features
    numeric = extract_sms_features(sms_clean)
    numeric = numeric[feature_columns]

    # Transform text using the original fitted TF-IDF vectorizer
    text_features = tfidf_new.transform([sms_clean])

    # Scale numerical features
    numeric_scaled = scaler_new.transform(numeric)

    # Combine into the original 5046-feature format
    final_features = hstack([
        text_features,
        numeric_scaled
    ]).tocsr()

    print("\nSMS:", sms)
    print("\nPREDICTIONS")
    print("-" * 55)

    for name, model in models_new.items():

        prediction = model.predict(final_features)[0]

        result = "PHISHING / SMISHING" if prediction == 1 else "LEGITIMATE"

        print(f"{name}: {result}")

    print("=" * 55)


test_sms_message()

MULTI-MODEL SMS PHISHING DETECTION



Enter SMS message:  URGENT! Your bank account is blocked. Verify your details immediately using this link http://example.com



SMS: URGENT! Your bank account is blocked. Verify your details immediately using this link http://example.com

PREDICTIONS
-------------------------------------------------------
Logistic Regression: PHISHING / SMISHING
Random Forest: PHISHING / SMISHING
SVM: PHISHING / SMISHING
XGBoost: PHISHING / SMISHING


In [146]:
import pandas as pd
import numpy as np
import re
import string
from scipy.sparse import hstack

def extract_sms_features(message):

    words = message.split()
    lower = message.lower()

    urls = re.findall(r'https?://\S+|www\.\S+', message)
    digits = sum(c.isdigit() for c in message)
    uppercase = sum(c.isupper() for c in message)
    special = sum(c in string.punctuation for c in message)

    urgent = ["urgent", "immediately", "now", "alert", "limited"]
    financial = ["bank", "account", "payment", "money", "refund"]
    credential = ["password", "otp", "pin", "verify", "login"]
    action = ["click", "call", "claim", "update", "confirm"]
    reward = ["win", "winner", "prize", "free", "reward"]
    delivery = ["delivery", "parcel", "package", "shipment"]
    suspicious = ["suspended", "blocked", "expire", "congratulations"]

    def count(items):
        return sum(lower.count(w) for w in items)

    def has(items):
        return int(any(w in lower for w in items))

    url_lengths = [len(u) for u in urls]

    email_count = len(re.findall(r'\b[\w.-]+@[\w.-]+\.\w+\b', message))
    phone_count = len(re.findall(r'\b\d{7,15}\b', message))
    ip_count = len(re.findall(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', message))

    features = {
        "message_length": len(message),
        "word_count": len(words),
        "avg_word_length": np.mean([len(w) for w in words]) if words else 0,
        "unique_word_ratio": len(set(words)) / len(words) if words else 0,
        "digit_count": digits,
        "digit_ratio": digits / len(message) if message else 0,
        "uppercase_count": uppercase,
        "uppercase_ratio": uppercase / len(message) if message else 0,
        "special_char_count": special,
        "special_char_ratio": special / len(message) if message else 0,
        "space_count": message.count(" "),
        "exclamation_count": message.count("!"),
        "question_count": message.count("?"),
        "currency_symbol_count": sum(c in "$£€₹" for c in message),
        "percent_count": message.count("%"),
        "has_all_caps_word": int(any(w.isupper() and len(w) > 1 for w in words)),
        "repeated_punctuation_count": len(re.findall(r'([!?.,])\1+', message)),
        "repeated_character_count": len(re.findall(r'(.)\1{2,}', message)),
        "leet_obfuscation_count": sum(lower.count(x) for x in ["0", "1", "3", "4", "5", "@"]),
        "url_count": len(urls),
        "has_url": int(bool(urls)),
        "url_char_count": sum(url_lengths),
        "avg_url_length": np.mean(url_lengths) if urls else 0,
        "shortened_url_count": sum(any(d in u.lower() for d in ["bit.ly", "tinyurl", "t.co", "goo.gl"]) for u in urls),
        "suspicious_tld_count": sum(any(t in u.lower() for t in [".xyz", ".top", ".click", ".loan"]) for u in urls),
        "ip_address_count": ip_count,
        "has_ip_address": int(ip_count > 0),
        "email_count": email_count,
        "has_email": int(email_count > 0),
        "phone_count": phone_count,
        "has_phone": int(phone_count > 0),
        "urgent_keyword_count": count(urgent),
        "financial_keyword_count": count(financial),
        "credential_keyword_count": count(credential),
        "action_keyword_count": count(action),
        "reward_keyword_count": count(reward),
        "delivery_keyword_count": count(delivery),
        "suspicious_keyword_count": count(suspicious),
        "has_urgent_language": has(urgent),
        "has_financial_language": has(financial),
        "has_credential_language": has(credential),
        "has_action_language": has(action),
        "has_reward_language": has(reward),
        "url_field_yes": int(bool(urls)),
        "email_field_yes": int(email_count > 0),
        "phone_field_yes": int(phone_count > 0)
    }

    return pd.DataFrame([features])


def test_sms_message():

    print("=" * 55)
    print("MULTI-MODEL SMS PHISHING DETECTION")
    print("=" * 55)

    sms = input("\nEnter SMS message: ").strip()

    if not sms:
        print("Please enter a message.")
        return

    sms_clean = sms.lower().strip()

    # Extract numerical features
    numeric = extract_sms_features(sms_clean)
    numeric = numeric[feature_columns]

    # Transform text using the original fitted TF-IDF vectorizer
    text_features = tfidf_new.transform([sms_clean])

    # Scale numerical features
    numeric_scaled = scaler_new.transform(numeric)

    # Combine into the original 5046-feature format
    final_features = hstack([
        text_features,
        numeric_scaled
    ]).tocsr()

    print("\nSMS:", sms)
    print("\nPREDICTIONS")
    print("-" * 55)

    for name, model in models_new.items():

        prediction = model.predict(final_features)[0]

        result = "PHISHING / SMISHING" if prediction == 1 else "LEGITIMATE"

        print(f"{name}: {result}")

    print("=" * 55)


test_sms_message()

MULTI-MODEL SMS PHISHING DETECTION



Enter SMS message:  Your mobile number has been selected for an exclusive reward. Call 08001234567 before 6 PM today to know your eligibility.



SMS: Your mobile number has been selected for an exclusive reward. Call 08001234567 before 6 PM today to know your eligibility.

PREDICTIONS
-------------------------------------------------------
Logistic Regression: PHISHING / SMISHING
Random Forest: PHISHING / SMISHING
SVM: PHISHING / SMISHING
XGBoost: PHISHING / SMISHING


In [147]:
import pandas as pd
import numpy as np
import re
import string
from scipy.sparse import hstack

def extract_sms_features(message):

    words = message.split()
    lower = message.lower()

    urls = re.findall(r'https?://\S+|www\.\S+', message)
    digits = sum(c.isdigit() for c in message)
    uppercase = sum(c.isupper() for c in message)
    special = sum(c in string.punctuation for c in message)

    urgent = ["urgent", "immediately", "now", "alert", "limited"]
    financial = ["bank", "account", "payment", "money", "refund"]
    credential = ["password", "otp", "pin", "verify", "login"]
    action = ["click", "call", "claim", "update", "confirm"]
    reward = ["win", "winner", "prize", "free", "reward"]
    delivery = ["delivery", "parcel", "package", "shipment"]
    suspicious = ["suspended", "blocked", "expire", "congratulations"]

    def count(items):
        return sum(lower.count(w) for w in items)

    def has(items):
        return int(any(w in lower for w in items))

    url_lengths = [len(u) for u in urls]

    email_count = len(re.findall(r'\b[\w.-]+@[\w.-]+\.\w+\b', message))
    phone_count = len(re.findall(r'\b\d{7,15}\b', message))
    ip_count = len(re.findall(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', message))

    features = {
        "message_length": len(message),
        "word_count": len(words),
        "avg_word_length": np.mean([len(w) for w in words]) if words else 0,
        "unique_word_ratio": len(set(words)) / len(words) if words else 0,
        "digit_count": digits,
        "digit_ratio": digits / len(message) if message else 0,
        "uppercase_count": uppercase,
        "uppercase_ratio": uppercase / len(message) if message else 0,
        "special_char_count": special,
        "special_char_ratio": special / len(message) if message else 0,
        "space_count": message.count(" "),
        "exclamation_count": message.count("!"),
        "question_count": message.count("?"),
        "currency_symbol_count": sum(c in "$£€₹" for c in message),
        "percent_count": message.count("%"),
        "has_all_caps_word": int(any(w.isupper() and len(w) > 1 for w in words)),
        "repeated_punctuation_count": len(re.findall(r'([!?.,])\1+', message)),
        "repeated_character_count": len(re.findall(r'(.)\1{2,}', message)),
        "leet_obfuscation_count": sum(lower.count(x) for x in ["0", "1", "3", "4", "5", "@"]),
        "url_count": len(urls),
        "has_url": int(bool(urls)),
        "url_char_count": sum(url_lengths),
        "avg_url_length": np.mean(url_lengths) if urls else 0,
        "shortened_url_count": sum(any(d in u.lower() for d in ["bit.ly", "tinyurl", "t.co", "goo.gl"]) for u in urls),
        "suspicious_tld_count": sum(any(t in u.lower() for t in [".xyz", ".top", ".click", ".loan"]) for u in urls),
        "ip_address_count": ip_count,
        "has_ip_address": int(ip_count > 0),
        "email_count": email_count,
        "has_email": int(email_count > 0),
        "phone_count": phone_count,
        "has_phone": int(phone_count > 0),
        "urgent_keyword_count": count(urgent),
        "financial_keyword_count": count(financial),
        "credential_keyword_count": count(credential),
        "action_keyword_count": count(action),
        "reward_keyword_count": count(reward),
        "delivery_keyword_count": count(delivery),
        "suspicious_keyword_count": count(suspicious),
        "has_urgent_language": has(urgent),
        "has_financial_language": has(financial),
        "has_credential_language": has(credential),
        "has_action_language": has(action),
        "has_reward_language": has(reward),
        "url_field_yes": int(bool(urls)),
        "email_field_yes": int(email_count > 0),
        "phone_field_yes": int(phone_count > 0)
    }

    return pd.DataFrame([features])


def test_sms_message():

    print("=" * 55)
    print("MULTI-MODEL SMS PHISHING DETECTION")
    print("=" * 55)

    sms = input("\nEnter SMS message: ").strip()

    if not sms:
        print("Please enter a message.")
        return

    sms_clean = sms.lower().strip()

    # Extract numerical features
    numeric = extract_sms_features(sms_clean)
    numeric = numeric[feature_columns]

    # Transform text using the original fitted TF-IDF vectorizer
    text_features = tfidf_new.transform([sms_clean])

    # Scale numerical features
    numeric_scaled = scaler_new.transform(numeric)

    # Combine into the original 5046-feature format
    final_features = hstack([
        text_features,
        numeric_scaled
    ]).tocsr()

    print("\nSMS:", sms)
    print("\nPREDICTIONS")
    print("-" * 55)

    for name, model in models_new.items():

        prediction = model.predict(final_features)[0]

        result = "PHISHING / SMISHING" if prediction == 1 else "LEGITIMATE"

        print(f"{name}: {result}")

    print("=" * 55)


test_sms_message()

MULTI-MODEL SMS PHISHING DETECTION



Enter SMS message:  Dear Customer, a request to update your registered mobile number has been received. If this was not initiated by you, please verify your account immediately at http://secure-bank-check.example



SMS: Dear Customer, a request to update your registered mobile number has been received. If this was not initiated by you, please verify your account immediately at http://secure-bank-check.example

PREDICTIONS
-------------------------------------------------------
Logistic Regression: PHISHING / SMISHING
Random Forest: PHISHING / SMISHING
SVM: PHISHING / SMISHING
XGBoost: PHISHING / SMISHING


In [148]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Word-level TF-IDF
word_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.98
)

# Character-level TF-IDF
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=3000,
    min_df=2,
    sublinear_tf=True
)

# Transform training and internal testing text
X_train_word = word_vectorizer.fit_transform(X_train_text)
X_test_word = word_vectorizer.transform(X_test_text)

X_train_char = char_vectorizer.fit_transform(X_train_text)
X_test_char = char_vectorizer.transform(X_test_text)

# Combine word, character and numerical features
X_train_improved = hstack([
    X_train_word,
    X_train_char,
    X_train_num_scaled
]).tocsr()

X_test_improved = hstack([
    X_test_word,
    X_test_char,
    X_test_num_scaled
]).tocsr()

print("Improved training shape:", X_train_improved.shape)
print("Improved testing shape:", X_test_improved.shape)
print("Improved feature extraction completed!")

Improved training shape: (2224, 8046)
Improved testing shape: (556, 8046)
Improved feature extraction completed!


In [149]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier

models_improved = {
    "Logistic Regression": LogisticRegression(
        C=1.5,
        max_iter=5000,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "SVM": LinearSVC(
        C=1.5,
        class_weight="balanced",
        max_iter=15000,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
}

for name, model in models_improved.items():
    print("Training:", name)
    model.fit(X_train_improved, y_train)

print("\nAll four improved models trained successfully!")

Training: Logistic Regression
Training: Random Forest
Training: SVM
Training: XGBoost

All four improved models trained successfully!


In [150]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

improved_results = []

for name, model in models_improved.items():

    y_pred = model.predict(X_test_improved)

    improved_results.append({
        "Model": name,
        "Accuracy (%)": round(accuracy_score(y_test, y_pred) * 100, 2),
        "Precision (%)": round(precision_score(y_test, y_pred) * 100, 2),
        "Recall (%)": round(recall_score(y_test, y_pred) * 100, 2),
        "F1 Score (%)": round(f1_score(y_test, y_pred) * 100, 2)
    })

improved_results_df = pd.DataFrame(improved_results)

print("IMPROVED MODEL PERFORMANCE")
display(improved_results_df)

IMPROVED MODEL PERFORMANCE


,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%)
0,Logistic Regression,89.21,88.11,90.65,89.36
1,Random Forest,87.59,85.67,90.29,87.92
2,SVM,89.39,87.63,91.73,89.63
3,XGBoost,88.49,85.91,92.09,88.89


In [151]:
# Prepare external dataset using improved features

X_external_word = word_vectorizer.transform(
    df_external_binary["TEXT_CLEAN"]
)

X_external_char = char_vectorizer.transform(
    df_external_binary["TEXT_CLEAN"]
)

X_external_num_scaled = scaler_new.transform(
    df_external_binary[feature_columns].astype(float)
)

X_external_improved = hstack([
    X_external_word,
    X_external_char,
    X_external_num_scaled
]).tocsr()

print("External improved feature shape:", X_external_improved.shape)
print("External feature preparation completed!")

External improved feature shape: (6794, 8046)
External feature preparation completed!


In [152]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix
)
import pandas as pd

external_improved_results = []

for name, model in models_improved.items():

    y_pred = model.predict(X_external_improved)

    external_improved_results.append({
        "Model": name,
        "Accuracy (%)": round(
            accuracy_score(y_external_new, y_pred) * 100, 2
        ),
        "Precision (%)": round(
            precision_score(y_external_new, y_pred) * 100, 2
        ),
        "Recall (%)": round(
            recall_score(y_external_new, y_pred) * 100, 2
        ),
        "F1 Score (%)": round(
            f1_score(y_external_new, y_pred) * 100, 2
        )
    })

    print("\n", name)
    print(confusion_matrix(y_external_new, y_pred))

external_improved_df = pd.DataFrame(external_improved_results)

print("\nIMPROVED EXTERNAL MODEL PERFORMANCE")
display(external_improved_df)


 Logistic Regression
[[2805  592]
 [1946 1451]]

 Random Forest
[[2945  452]
 [ 786 2611]]

 SVM
[[2621  776]
 [1965 1432]]

 XGBoost
[[2908  489]
 [ 562 2835]]

IMPROVED EXTERNAL MODEL PERFORMANCE


,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%)
0,Logistic Regression,62.64,71.02,42.71,53.35
1,Random Forest,81.78,85.24,76.86,80.84
2,SVM,59.66,64.86,42.15,51.10
3,XGBoost,84.53,85.29,83.46,84.36


In [153]:
from sklearn.metrics import classification_report

xgb_external_pred = models_improved["XGBoost"].predict(
    X_external_improved
)

print(classification_report(
    y_external_new,
    xgb_external_pred,
    target_names=["Legitimate", "Phishing"]
))

              precision    recall  f1-score   support

  Legitimate       0.84      0.86      0.85      3397
    Phishing       0.85      0.83      0.84      3397

    accuracy                           0.85      6794
   macro avg       0.85      0.85      0.85      6794
weighted avg       0.85      0.85      0.85      6794



In [154]:
from sklearn.model_selection import train_test_split

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train_improved,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Training shape:", X_fit.shape)
print("Validation shape:", X_val.shape)

Training shape: (1779, 8046)
Validation shape: (445, 8046)


In [155]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

xgb_tuned = XGBClassifier(
    n_estimators=700,
    max_depth=4,
    learning_rate=0.03,
    min_child_weight=2,
    subsample=0.85,
    colsample_bytree=0.85,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=2,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_tuned.fit(X_fit, y_fit)

y_val_pred = xgb_tuned.predict(X_val)

print("TUNED XGBOOST VALIDATION RESULTS")
print("Accuracy:", round(accuracy_score(y_val, y_val_pred) * 100, 2), "%")
print("Precision:", round(precision_score(y_val, y_val_pred) * 100, 2), "%")
print("Recall:", round(recall_score(y_val, y_val_pred) * 100, 2), "%")
print("F1 Score:", round(f1_score(y_val, y_val_pred) * 100, 2), "%")

TUNED XGBOOST VALIDATION RESULTS
Accuracy: 88.09 %
Precision: 85.65 %
Recall: 91.44 %
F1 Score: 88.45 %


In [156]:
y_test_tuned_pred = xgb_tuned.predict(X_test_improved)

print("TUNED XGBOOST INTERNAL TEST RESULTS")
print("Accuracy:", round(accuracy_score(y_test, y_test_tuned_pred) * 100, 2), "%")
print("Precision:", round(precision_score(y_test, y_test_tuned_pred) * 100, 2), "%")
print("Recall:", round(recall_score(y_test, y_test_tuned_pred) * 100, 2), "%")
print("F1 Score:", round(f1_score(y_test, y_test_tuned_pred) * 100, 2), "%")

TUNED XGBOOST INTERNAL TEST RESULTS
Accuracy: 87.59 %
Precision: 85.67 %
Recall: 90.29 %
F1 Score: 87.92 %


In [157]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

models_linear_tuned = {
    "Logistic Regression Tuned": LogisticRegression(
        C=0.5,
        max_iter=5000,
        class_weight="balanced"
    ),

    "SVM Tuned": LinearSVC(
        C=0.5,
        class_weight="balanced",
        max_iter=20000,
        random_state=42
    )
}

for name, model in models_linear_tuned.items():
    model.fit(X_train_improved, y_train)

    pred = model.predict(X_test_improved)

    print("\n", name)
    print("Accuracy:", round(accuracy_score(y_test, pred) * 100, 2), "%")
    print("Precision:", round(precision_score(y_test, pred) * 100, 2), "%")
    print("Recall:", round(recall_score(y_test, pred) * 100, 2), "%")
    print("F1 Score:", round(f1_score(y_test, pred) * 100, 2), "%")


 Logistic Regression Tuned
Accuracy: 86.33 %
Precision: 84.83 %
Recall: 88.49 %
F1 Score: 86.62 %

 SVM Tuned
Accuracy: 89.75 %
Precision: 88.5 %
Recall: 91.37 %
F1 Score: 89.91 %


In [158]:
svm_external_pred = models_linear_tuned["SVM Tuned"].predict(
    X_external_improved
)

print("TUNED SVM EXTERNAL RESULTS")
print("Accuracy:", round(accuracy_score(y_external_new, svm_external_pred) * 100, 2), "%")
print("Precision:", round(precision_score(y_external_new, svm_external_pred) * 100, 2), "%")
print("Recall:", round(recall_score(y_external_new, svm_external_pred) * 100, 2), "%")
print("F1 Score:", round(f1_score(y_external_new, svm_external_pred) * 100, 2), "%")

TUNED SVM EXTERNAL RESULTS
Accuracy: 60.38 %
Precision: 66.45 %
Recall: 41.92 %
F1 Score: 51.41 %


In [159]:
rf_tuned = RandomForestClassifier(
    n_estimators=600,
    max_depth=30,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train_improved, y_train)

rf_tuned_pred = rf_tuned.predict(X_test_improved)

print("TUNED RANDOM FOREST RESULTS")
print("Accuracy:", round(accuracy_score(y_test, rf_tuned_pred) * 100, 2), "%")
print("Precision:", round(precision_score(y_test, rf_tuned_pred) * 100, 2), "%")
print("Recall:", round(recall_score(y_test, rf_tuned_pred) * 100, 2), "%")
print("F1 Score:", round(f1_score(y_test, rf_tuned_pred) * 100, 2), "%")

TUNED RANDOM FOREST RESULTS
Accuracy: 86.33 %
Precision: 83.89 %
Recall: 89.93 %
F1 Score: 86.81 %


In [160]:

import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

# Select the best configurations from our experiments
final_models = {
    "Logistic Regression": models_improved["Logistic Regression"],
    "Random Forest": models_improved["Random Forest"],
    "SVM": models_linear_tuned["SVM Tuned"],
    "XGBoost": models_improved["XGBoost"]
}

final_results = []

for name, model in final_models.items():

    # Internal testing
    internal_pred = model.predict(X_test_improved)

    internal_accuracy = accuracy_score(y_test, internal_pred) * 100
    internal_precision = precision_score(y_test, internal_pred) * 100
    internal_recall = recall_score(y_test, internal_pred) * 100
    internal_f1 = f1_score(y_test, internal_pred) * 100

    # External testing
    external_pred = model.predict(X_external_improved)

    external_accuracy = accuracy_score(
        y_external_new, external_pred
    ) * 100

    external_precision = precision_score(
        y_external_new, external_pred
    ) * 100

    external_recall = recall_score(
        y_external_new, external_pred
    ) * 100

    external_f1 = f1_score(
        y_external_new, external_pred
    ) * 100

    final_results.append({
        "Model": name,
        "Internal Accuracy (%)": round(internal_accuracy, 2),
        "Internal Precision (%)": round(internal_precision, 2),
        "Internal Recall (%)": round(internal_recall, 2),
        "Internal F1 (%)": round(internal_f1, 2),
        "External Accuracy (%)": round(external_accuracy, 2),
        "External Precision (%)": round(external_precision, 2),
        "External Recall (%)": round(external_recall, 2),
        "External F1 (%)": round(external_f1, 2)
    })

final_results_df = pd.DataFrame(final_results)

print("=" * 100)
print("FINAL SMS PHISHING DETECTION MODEL PERFORMANCE")
print("=" * 100)

display(final_results_df)

# Identify best models
best_internal = final_results_df.loc[
    final_results_df["Internal Accuracy (%)"].idxmax()
]

best_external = final_results_df.loc[
    final_results_df["External Accuracy (%)"].idxmax()
]

print("\nBEST INTERNAL MODEL")
print("Model:", best_internal["Model"])
print("Accuracy:", best_internal["Internal Accuracy (%)"], "%")

print("\nBEST EXTERNAL MODEL")
print("Model:", best_external["Model"])
print("Accuracy:", best_external["External Accuracy (%)"], "%")

print("\n" + "=" * 100)


FINAL SMS PHISHING DETECTION MODEL PERFORMANCE


,Model,Internal Accuracy (%),Internal Precision (%),Internal Recall (%),Internal F1 (%),External Accuracy (%),External Precision (%),External Recall (%),External F1 (%)
0,Logistic Regression,89.21,88.11,90.65,89.36,62.64,71.02,42.71,53.35
1,Random Forest,87.59,85.67,90.29,87.92,81.78,85.24,76.86,80.84
2,SVM,89.75,88.50,91.37,89.91,60.38,66.45,41.92,51.41
3,XGBoost,88.49,85.91,92.09,88.89,84.53,85.29,83.46,84.36



BEST INTERNAL MODEL
Model: SVM
Accuracy: 89.75 %

BEST EXTERNAL MODEL
Model: XGBoost
Accuracy: 84.53 %



In [168]:

import pandas as pd

# INTERNAL DATASET RESULTS
internal_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "SVM (Tuned)",
        "XGBoost"
    ],
    "Accuracy (%)": [89.21, 87.59, 89.75, 88.49],
    "Precision (%)": [88.11, 85.67, 88.50, 85.91],
    "Recall (%)": [90.65, 90.29, 91.37, 92.09],
    "F1 Score (%)": [89.36, 87.92, 89.91, 88.89]
})

# EXTERNAL DATASET RESULTS
external_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "SVM (Tuned)",
        "XGBoost"
    ],
    "Accuracy (%)": [62.64, 81.78, 60.38, 84.53],
    "Precision (%)": [71.02, 85.24, 66.45, 85.29],
    "Recall (%)": [42.71, 76.86, 41.92, 83.46],
    "F1 Score (%)": [53.35, 80.84, 51.41, 84.36]
})

# DISPLAY INTERNAL RESULTS
print("=" * 75)
print("INTERNAL DATASET TESTING RESULTS")
print("=" * 75)
display(internal_results)

best_internal = internal_results.loc[
    internal_results["Accuracy (%)"].idxmax()
]

print("BEST INTERNAL MODEL:", best_internal["Model"])
print("BEST INTERNAL ACCURACY:", best_internal["Accuracy (%)"], "%")


# DISPLAY EXTERNAL RESULTS
print("\n" + "=" * 75)
print("EXTERNAL DATASET TESTING RESULTS")
print("=" * 75)
display(external_results)

best_external = external_results.loc[
    external_results["Accuracy (%)"].idxmax()
]

print("BEST EXTERNAL MODEL:", best_external["Model"])
print("BEST EXTERNAL ACCURACY:", best_external["Accuracy (%)"], "%")

print("\n" + "=" * 75)
print("FINAL MODEL EVALUATION COMPLETED")
print("=" * 75)


INTERNAL DATASET TESTING RESULTS


,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%)
0,Logistic Regression,89.21,88.11,90.65,89.36
1,Random Forest,87.59,85.67,90.29,87.92
2,SVM (Tuned),89.75,88.50,91.37,89.91
3,XGBoost,88.49,85.91,92.09,88.89


BEST INTERNAL MODEL: SVM (Tuned)
BEST INTERNAL ACCURACY: 89.75 %

EXTERNAL DATASET TESTING RESULTS


,Model,Accuracy (%),Precision (%),Recall (%),F1 Score (%)
0,Logistic Regression,62.64,71.02,42.71,53.35
1,Random Forest,81.78,85.24,76.86,80.84
2,SVM (Tuned),60.38,66.45,41.92,51.41
3,XGBoost,84.53,85.29,83.46,84.36


BEST EXTERNAL MODEL: XGBoost
BEST EXTERNAL ACCURACY: 84.53 %

FINAL MODEL EVALUATION COMPLETED


In [ ]:

import numpy as np
import pandas as pd
from scipy.sparse import hstack

def extract_sms_features(message):

    import re
    import string

    text = str(message)
    lower = text.lower()
    words = text.split()
    urls = re.findall(r'https?://\S+|www\.\S+', lower)

    urgent_words = [
        "urgent", "immediately", "now", "today", "limited",
        "expire", "deadline", "suspended", "blocked"
    ]

    financial_words = [
        "bank", "account", "payment", "money", "cash",
        "refund", "credit", "debit", "prize", "win", "reward"
    ]

    credential_words = [
        "password", "otp", "pin", "verify", "verification",
        "login", "credentials", "identity"
    ]

    action_words = [
        "click", "call", "reply", "submit", "update",
        "confirm", "visit", "open"
    ]

    reward_words = [
        "free", "won", "winner", "gift", "claim", "bonus"
    ]

    delivery_words = [
        "delivery", "parcel", "package", "shipment", "order"
    ]

    suspicious_words = [
        "urgent", "verify", "account", "password", "otp",
        "click", "free", "winner", "claim", "suspended",
        "blocked", "prize", "bank", "refund"
    ]

    def count_words(word_list):
        return sum(lower.count(word) for word in word_list)

    url_lengths = [len(url) for url in urls]

    features = {
        "message_length": len(text),
        "word_count": len(words),
        "avg_word_length": np.mean([len(w) for w in words]) if words else 0,
        "unique_word_ratio": len(set(words)) / len(words) if words else 0,
        "digit_count": sum(c.isdigit() for c in text),
        "digit_ratio": sum(c.isdigit() for c in text) / max(len(text), 1),
        "uppercase_count": sum(c.isupper() for c in text),
        "uppercase_ratio": sum(c.isupper() for c in text) / max(len(text), 1),
        "special_char_count": sum(c in string.punctuation for c in text),
        "special_char_ratio": sum(c in string.punctuation for c in text) / max(len(text), 1),
        "space_count": text.count(" "),
        "exclamation_count": text.count("!"),
        "question_count": text.count("?"),
        "currency_symbol_count": sum(c in "₹$£€" for c in text),
        "percent_count": text.count("%"),
        "has_all_caps_word": int(any(w.isupper() and len(w) > 1 for w in words)),
        "repeated_punctuation_count": len(re.findall(r'([!?.,])\1+', text)),
        "repeated_character_count": len(re.findall(r'(.)\1{2,}', lower)),
        "leet_obfuscation_count": sum(lower.count(x) for x in ["0", "1", "3", "4", "5", "@"]),
        "url_count": len(urls),
        "has_url": int(len(urls) > 0),
        "url_char_count": sum(url_lengths),
        "avg_url_length": np.mean(url_lengths) if url_lengths else 0,
        "shortened_url_count": sum(x in lower for x in ["bit.ly", "tinyurl", "t.co", "goo.gl"]),
        "suspicious_tld_count": sum(lower.count(x) for x in [".xyz", ".top", ".click", ".info"]),
        "ip_address_count": len(re.findall(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', lower)),
        "has_ip_address": int(bool(re.search(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', lower))),
        "email_count": len(re.findall(r'\b[\w.-]+@[\w.-]+\.\w+\b', lower)),
        "has_email": int(bool(re.search(r'\b[\w.-]+@[\w.-]+\.\w+\b', lower))),
        "phone_count": len(re.findall(r'\b\d{7,}\b', text)),
        "has_phone": int(bool(re.search(r'\b\d{7,}\b', text))),
        "urgent_keyword_count": count_words(urgent_words),
        "financial_keyword_count": count_words(financial_words),
        "credential_keyword_count": count_words(credential_words),
        "action_keyword_count": count_words(action_words),
        "reward_keyword_count": count_words(reward_words),
        "delivery_keyword_count": count_words(delivery_words),
        "suspicious_keyword_count": count_words(suspicious_words),
        "has_urgent_language": int(count_words(urgent_words) > 0),
        "has_financial_language": int(count_words(financial_words) > 0),
        "has_credential_language": int(count_words(credential_words) > 0),
        "has_action_language": int(count_words(action_words) > 0),
        "has_reward_language": int(count_words(reward_words) > 0),
        "url_field_yes": int(len(urls) > 0),
        "email_field_yes": int(bool(re.search(r'\b[\w.-]+@[\w.-]+\.\w+\b', lower))),
        "phone_field_yes": int(bool(re.search(r'\b\d{7,}\b', text)))
    }

    return pd.DataFrame([features])[feature_columns]


def test_sms():

    message = input("\nEnter an SMS message: ").strip()

    if not message:
        print("Please enter a valid SMS message.")
        return

    # Prepare text
    clean_message = message.lower().strip()

    # Extract numerical features
    numeric_features = extract_sms_features(message)

    # Transform using fitted vectorizers and scaler
    word_features = word_vectorizer.transform([clean_message])
    char_features = char_vectorizer.transform([clean_message])
    scaled_features = scaler_new.transform(numeric_features)

    # Combine all 8046 features
    final_features = hstack([
        word_features,
        char_features,
        scaled_features
    ]).tocsr()

    print("\n" + "=" * 65)
    print("SMS PHISHING DETECTION RESULTS")
    print("=" * 65)

    predictions = []

    for name, model in final_models.items():

        prediction = model.predict(final_features)[0]

        label = "PHISHING / SPAM" if prediction == 1 else "LEGITIMATE"

        predictions.append(label)

        print(f"\n{name}: {label}")

    phishing_votes = predictions.count("PHISHING / SPAM")
    legitimate_votes = predictions.count("LEGITIMATE")

    print("\n" + "-" * 65)
    print("FINAL MODEL VOTING RESULT")

    if phishing_votes > legitimate_votes:
        print("FINAL PREDICTION: PHISHING / SPAM")
    elif legitimate_votes > phishing_votes:
        print("FINAL PREDICTION: LEGITIMATE")
    else:
        print("FINAL PREDICTION: INCONCLUSIVE (2-2 VOTE)")

    print(f"Phishing votes: {phishing_votes}/4")
    print(f"Legitimate votes: {legitimate_votes}/4")

    print("=" * 65)


# Start testing
test_sms()
